In [1]:
import pandas as pd
import numpy as np

In [ ]:
billing = pd.read_csv(r'C:billing_and_costs_c.csv', parse_dates=['billing_date'])
admissions = pd.read_csv(r'C:admission_c.csv')
patients = pd.read_csv(r'C:patients_c.csv')
insurance = pd.read_csv(r'C:insurance_claims_c.csv', parse_dates=['claim_submitted_date', 'claim_approved_date'])

In [36]:
print(billing.shape)
print(billing.dtypes)
billing.head()

(13429, 12)
billing_id                         str
admission_id                       str
patient_id                         str
billing_date            datetime64[us]
total_treatment_cost           float64
medication_cost                float64
lab_cost                       float64
procedure_cost                 float64
revenue_collected              float64
insurance_payout               float64
out_of_pocket                  float64
payment_status                     str
dtype: object


,billing_id,admission_id,patient_id,billing_date,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket,payment_status
0,BIL-000001,ADM-002070,PAT-07844,2021-01-03,2415.02,539.51,376.28,917.53,1710.32,1353.91,356.41,Paid
1,BIL-000002,ADM-009484,PAT-06074,2019-07-01,4268.26,1474.63,995.55,1203.01,3104.19,2131.59,972.60,Paid
2,BIL-000003,ADM-006331,PAT-01784,2022-11-10,500.00,121.84,105.59,135.46,326.29,236.00,90.29,Paid
3,BIL-000004,ADM-001084,PAT-07425,2017-02-06,5622.23,1786.61,955.60,1839.23,4882.81,3219.19,1663.62,Paid
4,BIL-000005,ADM-009785,PAT-00566,2019-12-16,3560.19,764.20,777.63,1125.10,2757.04,2065.12,691.92,Paid


In [72]:
# Basic feature engineering: create derived columns
billing['year'] = billing['billing_date'].dt.year
billing['month'] = billing['billing_date'].dt.month
insurance['year'] = insurance['claim_submitted_date'].dt.year
insurance['month'] = insurance['claim_submitted_date'].dt.month


In [53]:
# Summary Statistics
billing[['total_treatment_cost','medication_cost','lab_cost','procedure_cost','revenue_collected','insurance_payout','out_of_pocket']].describe()

,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket
count,12355.000000,13429.000000,13429.000000,13429.000000,13429.000000,13429.000000,13429.000000
mean,5168.385554,1408.812553,1024.249978,1669.309179,3821.491183,2642.958091,1040.401386
std,3223.387964,907.456227,661.470810,1067.943202,2760.698392,2103.002105,718.218646
min,447.080000,100.610000,75.120000,125.050000,55.280000,0.000000,55.280000
25%,3007.215000,797.000000,579.630000,950.480000,1945.060000,1293.370000,559.890000
50%,4404.170000,1189.310000,864.240000,1420.840000,3196.580000,2300.710000,863.130000
75%,6495.285000,1780.330000,1298.880000,2096.590000,4961.580000,3594.990000,1323.900000
max,28289.240000,8498.330000,6881.700000,9471.290000,25272.270000,17760.210000,6700.280000


In [54]:
# Payment status breakdown
billing.groupby('payment_status')['total_treatment_cost'].agg(['count', 'sum', 'mean'])

,count,sum,mean
payment_status,,,
Paid,8314,42081031.84,5061.466423
Pending,2755,14715786.16,5341.483180
Written-Off,1286,7058585.52,5488.791229


In [55]:
# Total revenue & Revenue collection rate
print(billing[['total_treatment_cost', 'revenue_collected']].sum())
print()
billing['collection_rate'] = billing['revenue_collected'] / billing['total_treatment_cost']
print(billing['collection_rate'].median())

total_treatment_cost    63855403.52
revenue_collected       51318805.10
dtype: float64

0.7820846684549468


In [56]:
# Yearly revenue trend analyses
billing.groupby('year')[['total_treatment_cost','revenue_collected']].sum()

,total_treatment_cost,revenue_collected
year,,
2017,4640289.38,3675812.70
2018,4920570.03,4017726.47
2019,5882208.19,4722107.10
2020,17294162.94,13749454.73
2021,13390277.01,10746211.56
2022,8445774.15,6889539.01
2023,5281977.52,4284671.23
2024,4000144.30,3233282.30


In [57]:
# Avg. costs by departments
m = billing.merge(admissions[['admission_id','ward_department']], on='admission_id')
m.groupby('ward_department')[['total_treatment_cost','revenue_collected']].mean().sort_values('total_treatment_cost', ascending=False)

,total_treatment_cost,revenue_collected
ward_department,,
ICU,10312.370418,7580.208316
Oncology,7090.816979,5263.414737
Cardiology,6109.111660,4570.811716
Neurology,5260.285671,3860.487862
Pulmonology,5234.706837,3878.408158
Orthopedics,4624.235590,3367.474945
Maternity,4163.254390,3092.912050
Emergency,3757.499856,2755.304255
Pediatrics,3354.438020,2457.286018


In [73]:
# Cost analysis with outlier detection
billing['total_treatment_cost'].quantile([0.25, 0.5, 0.75, 0.90, 0.95, 0.99])

# Flag high-cost outliers (above 95th percentile)
threshold = billing['total_treatment_cost'].quantile(0.95)
high_cost = billing[billing['total_treatment_cost'] > p95]
high_cost['payment_status'].value_counts()

payment_status
Paid           393
Pending        149
Written-Off     76
Name: count, dtype: int64